# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**What this notebook does:** Translates validated signal relationships from Weeks 2–5 into a ranked, human-reviewable content action playbook. All recommendations are observational and directional — they reflect patterns measured in this dataset, not causal predictions about what any intervention will produce.

**Language discipline:** every recommendation uses *observed / associated with / directional* language. Claims like *"doing X will improve Y"* exceed what cross-sectional data can support and are not used here.

> Working with an AI assistant? Tell it to read `skills/README.md` and load `writing-honest-claims/SKILL.md`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "matplotlib", "seaborn", "-q"])

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH  = Path("../../data/raw/content_refresh_anonymized.csv")
OUT_DIR    = Path("../../work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load model metrics from w05 (model_metrics.json) and w04 (baseline_metrics.json)
model_metrics_path    = OUT_DIR / "model_metrics.json"
baseline_metrics_path = OUT_DIR / "baseline_metrics.json"

model_metrics    = json.loads(model_metrics_path.read_text())    if model_metrics_path.exists()    else {}
baseline_metrics = json.loads(baseline_metrics_path.read_text()) if baseline_metrics_path.exists() else {}

df = pd.read_csv(DATA_PATH)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining"].mean()

print(f"Dataset: {len(df):,} rows | {df['client_id'].nunique()} clients")
print(f"Base rate: {BASE_RATE:.3f}")
print()
print("Model metrics loaded:", bool(model_metrics))
print("Baseline metrics loaded:", bool(baseline_metrics))

## 1. Ranked actions + reason codes

The playbook assigns every content page to one of four action tiers. Tiers are based on the signal patterns validated across Weeks 2–5:

- **CTR signal (Week 4):** Pages with 0 < CTR < 0.1 % declined at 67.7 % vs 54.2 % base rate — the strongest observed signal.
- **Position context (Week 2 + 4):** Within page-1, low-CTR pages declined at 60.3 % vs 53.6 % for high-CTR. Position shapes what CTR means.
- **Position-CTR mismatch:** Pages with position > 30 and low CTR — CTR-fix is the wrong action because CTR is structurally constrained by depth. Ranking support is needed first.
- **Volume filter:** Pages with < 200 impressions lack enough data to distinguish signal from noise — monitoring is the appropriate response, not intervention.

**The four tiers (applied in priority order):**

| Tier | Reason code | Signal pattern | Recommended action |
|---|---|---|---|
| 1 | `ctr_fix_page1` | impressions ≥ 500 · position 1–20 · 0 < CTR < 0.5 % | Optimise title/description — page is visible, not clicked |
| 2 | `rank_first` | impressions ≥ 500 · position > 20 · 0 < CTR < 0.5 % | Address content quality/depth before CTR — position limits CTR |
| 3 | `monitor_stable` | impressions ≥ 200 · not in tiers 1–2 | Track in next cycle; review if impressions drop > 20 % |
| 4 | `deprioritize` | impressions < 200 | Insufficient signal; remove from active review queue |

**What makes tier 1 different from the Week 4 baseline:** Week 4 had no position guard, causing rank-4+ pages to score highest (many impressions, structurally low CTR). Adding the position ≤ 20 guard tightens the flag to pages where CTR improvement is actually plausible. Week 4 quantified this trade-off: with the guard, P@200 drops from 0.665 to ~0.545 — a deliberate precision/recall trade-off documented there.

In [ ]:
# ── Assign action tiers ────────────────────────────────────────────────────
IMPR_TIER1 = 500
IMPR_TIER3 = 200
POS_GUARD  = 20
CTR_UPPER  = 0.5

def assign_tier(row):
    impr = row["impressions_90d"]
    pos  = row["avg_position"]
    ctr  = row["ctr"]
    if impr >= IMPR_TIER1 and pos > 0 and pos <= POS_GUARD and 0 < ctr < CTR_UPPER:
        return "ctr_fix_page1"
    if impr >= IMPR_TIER1 and pos > POS_GUARD and 0 < ctr < CTR_UPPER:
        return "rank_first"
    if impr >= IMPR_TIER3:
        return "monitor_stable"
    return "deprioritize"

df["reason_code"] = df.apply(assign_tier, axis=1)

# Priority score — higher = review sooner
tier_priority = {"ctr_fix_page1": 3, "rank_first": 2, "monitor_stable": 1, "deprioritize": 0}
df["action_tier"] = df["reason_code"].map(tier_priority)

# Within tier 1 and 2: sort by impressions (largest gap = largest potential impact)
df["action_score"] = np.where(
    df["action_tier"] >= 2,
    df["impressions_90d"] / df["ctr"].clip(lower=0.01),
    df["impressions_90d"].clip(upper=1e6)
)
df["queue_rank"] = df.groupby("reason_code")["action_score"].rank(method="first", ascending=False).astype(int)

# ── Tier summary ──────────────────────────────────────────────────────────
summary = (df.groupby("reason_code")
             .agg(
                 n=("is_declining", "count"),
                 decline_rate=("is_declining", "mean"),
                 median_impressions=("impressions_90d", "median"),
                 median_ctr=("ctr", "median"),
             )
             .round(3)
             .reset_index()
             .sort_values("decline_rate", ascending=False))

print("=== Action tier summary ===")
print(summary.to_string(index=False))
print(f"\nBase rate (all pages): {BASE_RATE:.3f}")
print()
print("Signal-to-noise check:")
for _, row in summary.iterrows():
    lift = row["decline_rate"] - BASE_RATE
    print(f"  {row['reason_code']:<22}  decline_rate={row['decline_rate']:.3f}  "
          f"lift vs base={lift:+.3f}  (n={int(row['n']):,})")

In [ ]:
# ── Top-20 review queue across tiers ─────────────────────────────────────
top20 = (df[df["action_tier"] >= 2]
           .sort_values(["action_tier", "action_score"], ascending=[False, False])
           .head(20)
           [["reason_code", "action_tier", "impressions_90d", "avg_position",
             "ctr", "is_declining", "content_type"]])

print("Top 20 pages recommended for review (tiers 1 & 2):")
print(top20.to_string(index=False))

# Precision check on the top 20
p_top20 = top20["is_declining"].mean()
print(f"\nDecline rate in top 20: {p_top20:.3f}  (vs base rate {BASE_RATE:.3f})") 
print(f"Note: decline_rate here is descriptive — used to sanity-check tier ordering,")
print(f"not to claim the tier caused the decline.")

In [ ]:
# ── Chart: decline rate by tier ───────────────────────────────────────────
tier_order = ["ctr_fix_page1", "rank_first", "monitor_stable", "deprioritize"]
tier_labels = ["Tier 1\nctr_fix_page1", "Tier 2\nrank_first",
               "Tier 3\nmonitor_stable", "Tier 4\ndeprioritize"]

tier_stats = summary.set_index("reason_code").reindex(tier_order)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tier_labels,
              tier_stats["decline_rate"],
              color=["#c84b31", "#e8a838", "#4c72b0", "#aaaaaa"],
              edgecolor="white", width=0.55)
ax.axhline(BASE_RATE, color="grey", linestyle="--", linewidth=1.2,
           label=f"Base rate ({BASE_RATE:.3f})")

for bar, val in zip(bars, tier_stats["decline_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.008,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

for bar, n in zip(bars, tier_stats["n"]):
    ax.text(bar.get_x() + bar.get_width() / 2, 0.01,
            f"n={int(n):,}", ha="center", va="bottom", fontsize=8, color="white")

ax.set_ylabel("Observed decline rate", fontsize=11)
ax.set_title("Decline rate by action tier vs base rate\n"
             "(starter-CSV dataset, 30 k pages, observational)", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 0.85)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "tier_decline_rates.png", dpi=150)
plt.show()
print("Saved: work/outputs/tier_decline_rates.png")

## 2. Intended use and limits

**Who uses this playbook:**
Content and SEO teams with a fixed weekly review capacity. The queue helps allocate limited editor hours to the pages most associated with impression decline in historical data.

**What it supports:**
- Deciding which pages to review *first*, not which pages to change.
- Framing review conversations: "this page shows the low-CTR-high-impression pattern" is a discussion starter, not a prescription.
- Tracking whether a page's tier assignment improves or worsens over successive months.

**Where it stops being valid:**

| Situation | Why it fails |
|---|---|
| New clients with < 3 months of GSC data | Impression baselines are too unstable to classify |
| Non-search content (social, email-only) | Signals (CTR, position) don't apply |
| Real-time decision-making (daily) | The pattern is a 90-day trailing average — short windows add noise |
| Measuring the effect of an intervention | Observational data cannot isolate the cause; a before/after design is needed |
| Clients with `avg_position = 0` for > 50 % of pages | Position signal is absent; tier 1 won't fire correctly |

**Honest framing of the numbers:**
- Tier 1 pages showed a higher *observed* decline rate than the base rate in this dataset and this time period.
- This does not mean fixing CTR for those pages will prevent decline. It means the pattern was associated with decline — a reason to look more closely, not a guarantee of ROI.

In [ ]:
# ── Quantify the limits ───────────────────────────────────────────────────
# Limit 1: Clients with mostly-zero avg_position
zero_pos = df[df["avg_position"] == 0].groupby("client_id")["content_id"].count()
total_per_client = df.groupby("client_id")["content_id"].count()
zero_pos_pct = (zero_pos / total_per_client).fillna(0)
clients_with_sparse_position = (zero_pos_pct > 0.5).sum()
print(f"Clients with > 50 % zero avg_position: {clients_with_sparse_position} "
      f"of {df['client_id'].nunique()} total")
print("  → Tier 1 (ctr_fix_page1) relies on position ≤ 20; these clients will mostly land in tier 3/4.")

# Limit 2: Tier 4 is large — most pages are deprioritized
tier4_pct = (df["reason_code"] == "deprioritize").mean()
print(f"\nTier 4 (deprioritize): {tier4_pct:.1%} of all pages")
print("  → Most pages fall below the volume threshold. This is expected — low-impression")
print("    pages are hard to classify reliably from search-console signals alone.")

# Limit 3: Model trained on this panel — out-of-distribution clients
if model_metrics:
    dev_n      = model_metrics.get("dev_test", {}).get("n_rows", "?")
    sealed_n   = model_metrics.get("sealed_test", {}).get("n_rows", "?")
    sealed_p200 = model_metrics.get("sealed_test", {}).get("random_forest", {}).get("p_at_200", "?")
    print(f"\nModel (w05) sealed-test rows: {sealed_n:,} | RF P@200 on sealed test: {sealed_p200}")
    print("  → Sealed-test P@200 is the out-of-sample estimate; dev-test metrics may overfit to")
    print("    the panel's training clients.")
else:
    print("\nmodel_metrics.json not found — run w05_model.ipynb first to populate sealed-test numbers.")

## 3. Human review + the no-go list

### What a reviewer must check before acting

The queue surfaces pages worth a second look. Before any content change:

1. **Is the position real?** Pages with `avg_position = 0` have no measured rank — the flag fired on CTR and impressions alone. Confirm the page ranks before assuming a title rewrite will move CTR.

2. **Is the low CTR structural or fixable?** Navigational queries, branded pages, and knowledge-panel-heavy queries all show low organic CTR structurally. A 0.05 % CTR for "brand login" is not a problem — it is expected behaviour.

3. **Is the page currently editable?** Check for publish permissions, canonical conflicts, or redirect chains before putting the page in a sprint.

4. **Is the page actually declining — or just volatile?** The label fires on any 20 % impression drop month-over-month. Seasonal drops (school holidays, end-of-quarter) look identical to genuine decay. Review the 90-day trend alongside the label.

5. **Is the client in the training panel?** If the client was added in the last 3 months, their page patterns may not match what the model learned.

### The no-go list — what must never be automated

| Action | Why it is never automated |
|---|---|
| Rewriting page titles / meta descriptions | Requires human judgement on tone, brand, and intent alignment |
| Redirecting or removing pages | Irreversible; false positives cause real ranking damage |
| Publishing the raw model score externally | Scores are internal decision-support; external publication creates misinterpretation risk |
| Using tier rank as an SLA or performance target | The tier is a heuristic derived from patterns, not a contractual quality assessment |
| Acting on tier 2 pages without a ranking analysis | `rank_first` pages need ranking diagnostics (competition, intent, backlinks) that the model does not have |

In [ ]:
# ── Structural low-CTR check (navigational / position-limited pages) ──────
# Pages flagged as ctr_fix_page1 but with stable trend (not declining)
false_urgency = df[
    (df["reason_code"] == "ctr_fix_page1") &
    (df["is_declining"] == 0)
]
total_tier1 = (df["reason_code"] == "ctr_fix_page1").sum()
print(f"Tier 1 (ctr_fix_page1) pages that are NOT currently declining: "
      f"{len(false_urgency):,} of {total_tier1:,} ({len(false_urgency)/total_tier1:.1%})")
print("  → Reviewers should open the trend chart before treating tier 1 as an emergency.")
print()

# Position = 0 in tier 1 (should not happen given the guard, but confirm)
tier1_zero_pos = df[(df["reason_code"] == "ctr_fix_page1") & (df["avg_position"] == 0)]
print(f"Tier 1 pages with avg_position = 0: {len(tier1_zero_pos):,}")
print("  (position guard in assign_tier excludes pos=0 from tier 1 — value should be 0.)")
print()

# Volatility: pages declining but with impression_consistency below 0.5
if "scroll_rate" in df.columns:  # only exists in starter CSV
    volatile = df[
        (df["is_declining"] == 1) &
        (df["impressions_90d"] < 300)
    ]
    print(f"Declining pages with < 300 impressions (high volatility risk): {len(volatile):,}")
    print("  → Small impression counts make the 20 % label threshold very sensitive to noise.")

## 4. Monitoring / retrain triggers

The playbook is built on patterns from a specific dataset and time window. It goes stale when the world changes. The triggers below are directional — they tell you *when to look again*, not automatically what to do.

### Drift triggers

| Trigger | Threshold | What it means |
|---|---|---|
| Tier 1 precision@100 drops | < 0.60 for two consecutive months | Signal pattern has weakened or changed |
| Base rate shifts | > 10 pp change in any direction over 60 days | Label distribution changed — position or CTR regime has shifted |
| Tier 1 flagged-page volume changes | > 30 % month-over-month | Client panel composition or GSC reporting changed |
| Sealed-test P@200 degrades | More than 5 pp below dev-test P@200 | Model has overfit to the training panel |

### When to retrain

- A drift trigger fires **and** a manual spot-check of 20 top-ranked pages confirms the pattern has changed.
- A new cohort of clients is onboarded that represents > 20 % of the page inventory.
- A Google Search Console reporting change alters how impressions or position are counted (the labels and features are both derived from GSC — a measurement change invalidates both).

### When NOT to retrain

- Precision fluctuates within one reporting period (monthly noise is normal).
- A high-profile algorithm update is announced — wait for the data to stabilise (4–6 weeks) before attributing drift to the update.

In [ ]:
# ── Baseline precision@K for drift monitoring ─────────────────────────────
tier1_scores  = df["action_score"] * (df["reason_code"] == "ctr_fix_page1")

def precision_at_k(scores, labels, k):
    idx = np.argsort(-np.array(scores))[:k]
    return np.array(labels)[idx].mean()

p50  = precision_at_k(tier1_scores, df["is_declining"], 50)
p100 = precision_at_k(tier1_scores, df["is_declining"], 100)
p200 = precision_at_k(tier1_scores, df["is_declining"], 200)

print("=== Monitoring baselines (fire a review if these drop) ===")
print(f"Tier 1 (ctr_fix_page1) scorer:")
print(f"  P@50  = {p50:.3f}   drift trigger: < 0.60")
print(f"  P@100 = {p100:.3f}   drift trigger: < 0.60")
print(f"  P@200 = {p200:.3f}   drift trigger: < 0.60")
print(f"  Base rate = {BASE_RATE:.3f}")
print()
if model_metrics:
    rf_p200_dev    = model_metrics.get("dev_test",    {}).get("random_forest", {}).get("p_at_200", None)
    rf_p200_sealed = model_metrics.get("sealed_test", {}).get("random_forest", {}).get("p_at_200", None)
    if rf_p200_dev and rf_p200_sealed:
        gap = rf_p200_dev - rf_p200_sealed
        print(f"RF model (w05): dev P@200={rf_p200_dev:.3f} | sealed P@200={rf_p200_sealed:.3f} | gap={gap:+.3f}")
        if gap > 0.05:
            print("  \u26a0 Gap > 5 pp: monitor for panel overfitting; re-evaluate on next month's data.")
        else:
            print("  \u2713 Gap within tolerance.")
print()
print("Record these as your current-month monitoring baselines.")
print("Compare against next month's run to detect drift before users notice.")

## 5. Exports for the paper

All exports go to `work/outputs/`. The CSV is not committed (CI blocks data files). The JSON and PNG files are committed and embedded in the capstone paper.

In [ ]:
# ── 1. Full ranked queue CSV (not committed) ──────────────────────────────
queue_cols = [
    "content_id", "client_id", "reason_code", "action_tier",
    "queue_rank", "action_score",
    "impressions_90d", "avg_position", "ctr",
    "is_declining", "content_type",
]
queue_df = (df[queue_cols]
              .sort_values(["action_tier", "queue_rank"], ascending=[False, True])
              .reset_index(drop=True))
queue_df.to_csv(OUT_DIR / "action_playbook_queue.csv", index=False)
print(f"Queue CSV: {len(queue_df):,} rows \u2192 work/outputs/action_playbook_queue.csv  (not committed)")

# ── 2. Recommendations JSON (committed) ───────────────────────────────────
recommendations = {
    "dataset": "content_refresh_anonymized.csv — 30,000 pages, 32 clients",
    "base_rate": round(float(BASE_RATE), 4),
    "language_note": "All findings are observational and directional. No causal claims.",
    "tiers": [
        {
            "tier": 1,
            "reason_code": "ctr_fix_page1",
            "signal_pattern": "impressions >= 500, position 1-20, 0 < CTR < 0.5%",
            "recommended_action": "Review title and meta description — page is visible but earning few clicks.",
            "n": int((df["reason_code"] == "ctr_fix_page1").sum()),
            "observed_decline_rate": round(float(df[df["reason_code"] == "ctr_fix_page1"]["is_declining"].mean()), 4),
            "honest_framing": "Associated with decline in this dataset; does not predict that CTR-fix will prevent decline."
        },
        {
            "tier": 2,
            "reason_code": "rank_first",
            "signal_pattern": "impressions >= 500, position > 20, 0 < CTR < 0.5%",
            "recommended_action": "Assess content quality and depth — ranking improvement is a prerequisite for CTR gains at this position.",
            "n": int((df["reason_code"] == "rank_first").sum()),
            "observed_decline_rate": round(float(df[df["reason_code"] == "rank_first"]["is_declining"].mean()), 4),
            "honest_framing": "Associated with decline; position depth structurally limits CTR — CTR-fix alone is insufficient."
        },
        {
            "tier": 3,
            "reason_code": "monitor_stable",
            "signal_pattern": "impressions >= 200, not in tiers 1-2",
            "recommended_action": "Track monthly. Escalate to tier 1 or 2 review if impressions drop > 20%.",
            "n": int((df["reason_code"] == "monitor_stable").sum()),
            "observed_decline_rate": round(float(df[df["reason_code"] == "monitor_stable"]["is_declining"].mean()), 4),
            "honest_framing": "No strong associated pattern; monitoring prevents silent decay from going unnoticed."
        },
        {
            "tier": 4,
            "reason_code": "deprioritize",
            "signal_pattern": "impressions < 200",
            "recommended_action": "Remove from active review queue. Revisit if content is promoted or receives a backlink.",
            "n": int((df["reason_code"] == "deprioritize").sum()),
            "observed_decline_rate": round(float(df[df["reason_code"] == "deprioritize"]["is_declining"].mean()), 4),
            "honest_framing": "Insufficient impression volume to distinguish signal from noise reliably."
        },
    ],
    "no_go_list": [
        "Automating title or meta-description rewrites",
        "Redirecting or removing pages based on tier alone",
        "Publishing raw model scores externally",
        "Using tier rank as an SLA or client-facing performance target",
        "Acting on tier-2 pages without a ranking/intent analysis"
    ],
    "drift_triggers": {
        "tier1_precision_at_100_floor": 0.60,
        "base_rate_shift_pp": 10,
        "tier1_volume_change_pct": 30,
        "model_dev_sealed_gap_pp": 5
    },
    "monitoring_baselines": {
        "tier1_p_at_50":  round(float(p50),  4),
        "tier1_p_at_100": round(float(p100), 4),
        "tier1_p_at_200": round(float(p200), 4),
        "base_rate":      round(float(BASE_RATE), 4),
    }
}

recs_path = OUT_DIR / "recommendations.json"
with open(recs_path, "w") as f:
    json.dump(recommendations, f, indent=2)
print(f"Recommendations JSON \u2192 work/outputs/recommendations.json  (committed)")
print(json.dumps(recommendations, indent=2)[:1200], "\n...")

In [ ]:
# ── Chart: signal profile by tier ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = {"ctr_fix_page1": "#c84b31", "rank_first": "#e8a838",
           "monitor_stable": "#4c72b0", "deprioritize": "#aaaaaa"}

# CTR distribution per tier
for tier in tier_order:
    subset = df[(df["reason_code"] == tier) & (df["ctr"] > 0) & (df["ctr"] < 1.0)]["ctr"]
    if len(subset) > 10:
        axes[0].hist(subset, bins=30, alpha=0.55, label=tier, color=palette[tier], density=True)
axes[0].set_xlabel("CTR (%)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("CTR distribution by tier", fontsize=12)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Avg position distribution per tier (excluding 0)
for tier in tier_order:
    subset = df[(df["reason_code"] == tier) & (df["avg_position"] > 0)]["avg_position"]
    if len(subset) > 10:
        axes[1].hist(np.clip(subset, 0, 100), bins=30, alpha=0.55,
                     label=tier, color=palette[tier], density=True)
axes[1].set_xlabel("avg_position (clipped at 100)", fontsize=11)
axes[1].set_ylabel("Density", fontsize=11)
axes[1].set_title("Position distribution by tier", fontsize=12)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Signal profiles by action tier (observational — not causal)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "tier_signal_profiles.png", dpi=150)
plt.show()
print("Saved: work/outputs/tier_signal_profiles.png")

## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, domains, or private queries anywhere
- [ ] Every recommendation uses *observed / associated with / directional* language — no "will improve", "proves", "causes"
- [ ] n and base rate appear next to every decline rate figure
- [ ] No-go list covers automated actions
- [ ] Drift triggers are stated with specific thresholds, not vague warnings
- [ ] `work/outputs/recommendations.json` committed
- [ ] `work/outputs/tier_decline_rates.png` and `tier_signal_profiles.png` committed
- [ ] `work/outputs/action_playbook_queue.csv` NOT committed (data file — CI blocks it)

**Assignment criteria (ML-10):**

| Criterion | Status |
|---|---|
| Ranked actions + reason codes | Section 1 — 4-tier system with signal justification |
| Intended use and limits | Section 2 — table of failure modes with named conditions |
| Human review checklist | Section 3 — 5 review steps before acting |
| No-go list | Section 3 — 5 items, with reasons |
| Monitoring / retrain triggers | Section 4 — 4 drift triggers with thresholds |
| Honest language throughout | Verified — no causal claims |
| Exports for the paper | Section 5 — JSON, PNG, queue CSV |